In [1]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    DockArrivals,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
)

In [2]:
ubuntu_path = "/mnt/outer/Documents/vlzm/GFDRR_ubuntu/GFDRR/data/raw/202602-citibike-tripdata_1.csv"
mac_path = "/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv"

# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path=mac_path,
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), scale_capacity_factor = 10)

historical_flows_df_raw = graph_data.historical_flows_df.copy()

/Users/vladislav/Documents/vlzm/GFDRR/gbp/loaders/dataloader_raw.py:221: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.stations_capacities_df['capacity'] = 100


In [44]:
historical_flows_df_raw

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,start_period,planned_end_period,realized_end_period,resource_id,quantity,reason
0,hist_176292,0,0,0,user_trip,departed,classic_bike,6626.01,5703.13,<NA>,0,17,<NA>,<NA>,1,<NA>
1,hist_22891,0,0,4,user_trip,departed,classic_bike,6257.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>
2,hist_24113,0,0,4,user_trip,departed,classic_bike,6224.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>
3,hist_81565,0,0,4,user_trip,departed,classic_bike,7599.09,7599.02,<NA>,4,29,<NA>,<NA>,1,<NA>
4,hist_40992,0,0,5,user_trip,departed,classic_bike,6030.04,6339.06,<NA>,5,28,<NA>,<NA>,1,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1992771,hist_182635,0,1,359,user_trip,arrived,classic_bike,5303.08,6004.06,6004.06,342,359,359,<NA>,1,<NA>
1992772,hist_87216,0,1,359,user_trip,arrived,classic_bike,7688.12,7872.03,7872.03,343,359,359,<NA>,1,<NA>
1992773,hist_912617,0,1,359,user_trip,arrived,classic_bike,5581.01,5623.03,5623.03,338,359,359,<NA>,1,<NA>
1992774,hist_920108,0,1,359,user_trip,arrived,classic_bike,5581.01,5623.03,5623.03,338,359,359,<NA>,1,<NA>


In [7]:
# // graph_data.periods_df
# // graph_data.commodities_categories_df
# // graph_data.facilities_df

# // мне нужно сделать датафрейм в котором будут все возможные комбинации периодов, категорий товаров и объектов.
inventory_backbone = graph_data.periods_df[['period_id']].merge(graph_data.commodities_categories_df, how='cross').merge(graph_data.facilities_df[['facility_id']], how='cross')
inventory_backbone

,period_id,commodity_category,facility_id
0,0,classic_bike,6602.05
1,0,classic_bike,5311.08
2,0,classic_bike,6789.08
3,0,classic_bike,6605.08
4,0,classic_bike,5584.04
...,...,...,...
1627195,359,electric_bike,depot_6
1627196,359,electric_bike,depot_7
1627197,359,electric_bike,depot_8
1627198,359,electric_bike,depot_9


In [ ]:
inventory_backbone = (
    graph_data.periods_df[['period_id']]
    .merge(graph_data.commodities_categories_df, how='cross')
    .merge(graph_data.facilities_df[['facility_id']], how='cross')
)

historical_inventory_df_raw = graph_data.historical_inventory_df.copy()
historical_inventory_df = historical_inventory_df_raw.rename(columns={'quantity': 'quantity_eop'})

# Known starting stock (before period 0), per facility/commodity.
initial_inventory = graph_data.initial_inventory_df.rename(columns={'quantity': 'quantity_initial'})

# Complete panel: every period x commodity x facility, plus the starting stock.
historical_inventory_df = pd.merge(
    inventory_backbone, historical_inventory_df,
    on=['period_id', 'facility_id', 'commodity_category'], how='left',
)
historical_inventory_df = historical_inventory_df.merge(
    initial_inventory, on=['facility_id', 'commodity_category'], how='left',
)
historical_inventory_df = historical_inventory_df.sort_values(
    ['facility_id', 'commodity_category', 'period_id']
)

# A missing record means "stock did not change", not "stock is zero".
eop = historical_inventory_df.groupby(['facility_id', 'commodity_category'])['quantity_eop'].ffill()
# Leading gap (before the first record): fall back to the starting stock,
# then to 0 for pairs that never held anything.
historical_inventory_df['quantity_eop'] = (
    eop.fillna(historical_inventory_df['quantity_initial']).fillna(0)
)

# Start of period = end of the previous period; for period 0 it is the
# starting stock, not the end-of-period-0 value.
sop = historical_inventory_df.groupby(['facility_id', 'commodity_category'])['quantity_eop'].shift(1)
historical_inventory_df['quantity_sop'] = (
    sop.fillna(historical_inventory_df['quantity_initial']).fillna(0)
)

historical_inventory_df = historical_inventory_df.drop(columns='quantity_initial').reset_index(drop=True)
historical_inventory_df



,period_id,commodity_category,facility_id,quantity_eop,quantity_sop
0,0,classic_bike,1234.56,11.0,11.0
1,1,classic_bike,1234.56,11.0,11.0
2,2,classic_bike,1234.56,11.0,11.0
3,3,classic_bike,1234.56,11.0,11.0
4,4,classic_bike,1234.56,11.0,11.0
...,...,...,...,...,...
1627195,355,electric_bike,depot_9,0.0,0.0
1627196,356,electric_bike,depot_9,0.0,0.0
1627197,357,electric_bike,depot_9,0.0,0.0
1627198,358,electric_bike,depot_9,0.0,0.0


In [40]:
def build_historical_inventory_wiled_df(
    periods_df,
    commodities_categories_df,
    facilities_df,
    historical_inventory_df,
    initial_inventory_df,
):
    """Build a complete inventory panel from the inventory inputs.

    Returns one row per (period, commodity_category, facility) with the
    end-of-period and start-of-period stock filled in. A missing source
    record means "stock did not change", not "stock is zero".

    Parameters
    ----------
    periods_df : pandas.DataFrame
        Must contain ``period_id``.
    commodities_categories_df : pandas.DataFrame
        Must contain ``commodity_category``.
    facilities_df : pandas.DataFrame
        Must contain ``facility_id``.
    historical_inventory_df : pandas.DataFrame
        Recorded end-of-period stock: ``period_id``, ``facility_id``,
        ``commodity_category``, ``quantity``.
    initial_inventory_df : pandas.DataFrame
        Starting stock (before period 0): ``facility_id``,
        ``commodity_category``, ``quantity``.

    Returns
    -------
    pandas.DataFrame
        Columns: ``period_id``, ``commodity_category``, ``facility_id``,
        ``quantity_eop``, ``quantity_sop``.
    """
    # Full panel: every period x commodity x facility.
    inventory_backbone = (
        periods_df[['period_id']]
        .merge(commodities_categories_df, how='cross')
        .merge(facilities_df[['facility_id']], how='cross')
    )

    historical = historical_inventory_df.rename(columns={'quantity': 'quantity_eop'})
    # Known starting stock (before period 0), per facility/commodity.
    initial = initial_inventory_df.rename(columns={'quantity': 'quantity_initial'})

    df = pd.merge(
        inventory_backbone, historical,
        on=['period_id', 'facility_id', 'commodity_category'], how='left',
    )
    df = df.merge(
        initial, on=['facility_id', 'commodity_category'], how='left',
    )
    df = df.sort_values(['facility_id', 'commodity_category', 'period_id'])

    # A missing record means "stock did not change", not "stock is zero".
    eop = df.groupby(['facility_id', 'commodity_category'])['quantity_eop'].ffill()
    # Leading gap (before the first record): fall back to the starting stock,
    # then to 0 for pairs that never held anything.
    df['quantity_eop'] = eop.fillna(df['quantity_initial']).fillna(0)

    # Start of period = end of the previous period; for period 0 it is the
    # starting stock, not the end-of-period-0 value.
    sop = df.groupby(['facility_id', 'commodity_category'])['quantity_eop'].shift(1)
    df['quantity_sop'] = sop.fillna(df['quantity_initial']).fillna(0)

    df = df.drop(columns='quantity_initial').reset_index(drop=True)
    return df


historical_inventory_wiled_df = build_historical_inventory_wiled_df(
    periods_df=graph_data.periods_df,
    commodities_categories_df=graph_data.commodities_categories_df,
    facilities_df=graph_data.facilities_df,
    historical_inventory_df=graph_data.historical_inventory_df,
    initial_inventory_df=graph_data.initial_inventory_df,
)
historical_inventory_wiled_df

,period_id,commodity_category,facility_id,quantity_eop,quantity_sop
0,0,classic_bike,1234.56,11.0,11.0
1,1,classic_bike,1234.56,11.0,11.0
2,2,classic_bike,1234.56,11.0,11.0
3,3,classic_bike,1234.56,11.0,11.0
4,4,classic_bike,1234.56,11.0,11.0
...,...,...,...,...,...
1627195,355,electric_bike,depot_9,0.0,0.0
1627196,356,electric_bike,depot_9,0.0,0.0
1627197,357,electric_bike,depot_9,0.0,0.0
1627198,358,electric_bike,depot_9,0.0,0.0


In [42]:
graph_data.historical_departures_df

,period_id,facility_id,commodity_category,quantity
0,0,6626.01,classic_bike,1
1,4,6224.06,classic_bike,1
2,4,6257.06,classic_bike,1
3,4,7599.09,classic_bike,1
4,5,6030.04,classic_bike,1
...,...,...,...,...
403201,345,8782.01,classic_bike,1
403202,345,8782.01,electric_bike,1
403203,345,8795.01,electric_bike,1
403204,345,8879.02,electric_bike,1


In [43]:
graph_data.historical_arrivals_df

,period_id,facility_id,commodity_category,quantity
0,15,2698.07,electric_bike,1
1,15,2708.07,electric_bike,1
2,15,2843.13,electric_bike,1
3,15,2898.01,classic_bike,1
4,15,2898.01,electric_bike,1
...,...,...,...,...
401006,358,6560.01,classic_bike,6
401007,358,7020.02,classic_bike,1
401008,359,5623.03,classic_bike,3
401009,359,6004.06,classic_bike,1


In [29]:
historical_inventory_df[historical_inventory_df['quantity_eop'] > historical_inventory_df['quantity_sop']]

,period_id,commodity_category,facility_id,quantity_eop,quantity_sop
58,58,classic_bike,1234.56,12.0,11.0
101,101,classic_bike,1234.56,13.0,12.0
132,132,classic_bike,1234.56,14.0,13.0
173,173,classic_bike,1234.56,15.0,14.0
176,176,classic_bike,1234.56,15.0,14.0
...,...,...,...,...,...
1618788,228,classic_bike,SYS038,11.0,10.0
1618808,248,classic_bike,SYS038,11.0,10.0
1618812,252,classic_bike,SYS038,11.0,10.0
1619061,141,electric_bike,SYS038,11.0,10.0


In [34]:
historical_inventory_df[(historical_inventory_df['period_id'].isin([56,57,58,59,60])) & (historical_inventory_df['facility_id'] == '1234.56')]

,period_id,commodity_category,facility_id,quantity_eop,quantity_sop
56,56,classic_bike,1234.56,11.0,11.0
57,57,classic_bike,1234.56,11.0,11.0
58,58,classic_bike,1234.56,12.0,11.0
59,59,classic_bike,1234.56,12.0,12.0
60,60,classic_bike,1234.56,12.0,12.0
416,56,electric_bike,1234.56,16.0,16.0
417,57,electric_bike,1234.56,16.0,16.0
418,58,electric_bike,1234.56,16.0,16.0
419,59,electric_bike,1234.56,16.0,16.0
420,60,electric_bike,1234.56,16.0,16.0


In [21]:
historical_inventory_df[historical_inventory_df['period_id'] == 0]

,period_id,commodity_category,facility_id,quantity_end_of_period,quantity_start_of_period
1918,0,classic_bike,1234.56,11.0,11.0
4178,0,electric_bike,1234.56,14.0,14.0
2167,0,classic_bike,1964.01,11.0,11.0
4427,0,electric_bike,1964.01,13.0,13.0
2048,0,classic_bike,2009.04,12.0,12.0
...,...,...,...,...,...
4516,0,electric_bike,depot_7,0.0,0.0
2257,0,classic_bike,depot_8,0.0,0.0
4517,0,electric_bike,depot_8,0.0,0.0
2258,0,classic_bike,depot_9,0.0,0.0


In [10]:
historical_inventory_df_raw

,period_id,facility_id,commodity_category,quantity
0,0,1234.56,classic_bike,11
1,1,1234.56,classic_bike,11
2,2,1234.56,classic_bike,11
3,3,1234.56,classic_bike,11
4,4,1234.56,classic_bike,11
...,...,...,...,...
1606315,355,Shop Morgan,electric_bike,12
1606316,356,Shop Morgan,electric_bike,12
1606317,357,Shop Morgan,electric_bike,12
1606318,358,Shop Morgan,electric_bike,12
